# Bronze Transactions Ingestion — Azure Databricks

## Overview

This notebook implements incremental ingestion of banking transaction CSV files from Azure Data Lake Storage Gen2 into a Unity Catalog-managed Bronze Delta table using Databricks Auto Loader.

**Source:** `/Volumes/banking_project/bronze/banking_raw/transactions/`

**Target:** `banking_project.bronze.bronze_transactions`

**Key Features:**
- Incremental file ingestion using Auto Loader
- Schema inference and schema evolution handling
- Rescued data for unexpected source columns
- Checkpointing for incremental processing
- Ingestion metadata for traceability
- Bronze record-count validation

**Note:** Change Data Feed (CDF) was subsequently enabled on the Bronze Delta table to support downstream incremental processing.

## 1. Configuration

Define the source location, schema metadata location, checkpoint location, and target Bronze Delta table.

- `source_path`: ADLS transaction files accessed through a Unity Catalog Volume.
- `schema_path`: Stores Auto Loader schema metadata.
- `checkpoint_path`: Tracks streaming progress and processed files.
- `target_table`: Destination Bronze Delta table.

These paths are reused across ingestion runs to support incremental processing.

In [0]:

source_path = "/Volumes/banking_project/bronze/banking_raw/transactions/"

schema_path = "/Volumes/banking_project/bronze/autoloader_metadata/schemas/transactions/"

checkpoint_path = "/Volumes/banking_project/bronze/autoloader_metadata/checkpoints/transactions/"

target_table = "banking_project.bronze.bronze_transactions"


## 2. Incremental File Ingestion Using Auto Loader

Read banking transaction CSV files incrementally using Databricks Auto Loader.

**Configuration:**
- `cloudFiles`: Enables Auto Loader.
- `cloudFiles.format = csv`: Specifies the source file format.
- `header = true`: Uses the first CSV row as column names.
- `cloudFiles.schemaLocation`: Stores inferred schema information.
- `cloudFiles.schemaEvolutionMode = rescue`: Preserves unexpected data in `_rescued_data`.
- `cloudFiles.useManagedFileEvents = false`: Disables managed file events for this ingestion.

**Additional Metadata:**
- `ingestion_timestamp`: Records when data is processed.
- `source_file`: Captures the source file path.

**Purpose:** Incrementally ingest source transactions while preserving raw values and ingestion traceability.

In [0]:

from pyspark.sql.functions import current_timestamp, col

df_transactions = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("cloudFiles.useManagedFileEvents", "false")
    .load(source_path)
)

df_bronze = (
    df_transactions
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)


## 3. Writing Transactions to the Bronze Delta Table

Write incoming transactions into the Unity Catalog Bronze Delta table using Structured Streaming.

**Configuration:**
- Delta Lake storage format.
- Checkpointing to track streaming progress.
- `availableNow=True` to process currently available files and stop.
- `toTable()` to write records to the Bronze table.

**Incremental Processing:**

Auto Loader identifies newly available files using its existing checkpoint, avoiding unnecessary reprocessing of previously ingested files.

**Purpose:** Maintain a persistent Bronze table containing raw transaction records for downstream Silver transformations.

In [0]:

query = (
    df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()


## 4. Bronze Data Validation

Verify that transaction records have been successfully ingested into the Bronze Delta table.

**Validation Query:**

Count the total number of records stored in the Bronze table.

**Observed Results:**
- Initial ingestion: 10,000 records
- Batch 04: 100 additional records
- Batch 05: 10 additional records
- Batch 06: 15 additional records
- Total Bronze records: 10,125

**Purpose:** Confirm successful incremental ingestion and verify that the expected transaction volume is available for downstream processing.

**Note:** Bronze preserves raw records, including values that may fail Silver data quality checks.

In [0]:
spark.sql("""
    SELECT COUNT(*) AS total_records
    FROM banking_project.bronze.bronze_transactions
""").show()

In [0]:
%sql DESCRIBE HISTORY banking_project.bronze.bronze_transactions;

## 5. Change Data Feed Configuration

Change Data Feed (CDF) was enabled on the Bronze Delta table to support incremental downstream processing.

CDF exposes row-level changes committed after the feature was enabled.

**Configuration Command:**

ALTER TABLE banking_project.bronze.bronze_transactions
SET TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);

**Validation:**
- CDF was enabled before Batch 06 ingestion.
- Delta version 5 contained 15 inserted records.
- CDF successfully exposed the 15 changes.

**Purpose:** Allow the Silver pipeline to read new Bronze changes without repeatedly processing the full Bronze dataset.

In [0]:
%sql
SELECT
    transaction_id,
    customer_id,
    transaction_amount,
    _change_type,
    _commit_version
FROM table_changes(
    'banking_project.bronze.bronze_transactions',
    5,
    5
);